# SmartBin: training notebook

Trains the VGG16 experiments E1-E4 and the MobileNetV2 comparison E5 on TrashNet.

**Before you run this:**
1. Runtime > Change runtime type > **T4 GPU**.
2. Copy `model.py`, `data.py`, `train.py`, `report.py`, `run_experiments.py` and `splits.csv` to `MyDrive/smartbin` in Google Drive. **Don't copy `results/`**: experiments already listed in `results/results.csv` are skipped, so nothing would train.

Results are saved to Drive after every experiment. If Colab disconnects, run all the cells again: finished experiments are skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/smartbin")
sys.path.insert(0, str(PROJECT))

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Data: download TrashNet and make the 70/15/15 split (seed 42)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from data import download_trashnet, make_loaders, make_splits
from model import CLASSES, EXPERIMENTS

root = download_trashnet(Path("/content/data"))
df = make_splits(root, PROJECT / "splits.csv")
print(df.split.value_counts())

fig, axes = plt.subplots(4, 6, figsize=(12, 7))
for col, name in enumerate(CLASSES):
    samples = df[df.label == col].sample(4, random_state=42)
    for row, path in enumerate(samples.path):
        axes[row, col].imshow(Image.open(root / path))
        axes[row, col].axis("off")
    axes[0, col].set_title(name)
fig

## 2. Sanity check: E2 should nearly memorise one batch

In [ ]:
from run_experiments import overfit_check

acc = overfit_check("E2", make_loaders(df, root), device)
print(f"one-batch accuracy: {acc:.2f}")
assert acc >= 0.8, "The training loop can't memorise one batch. Fix this before training."

## 3. Run the experiments (about 3-6 minutes each)

In [ ]:
from run_experiments import run_all

df = run_all(root, PROJECT, EXPERIMENTS, epochs=10, device=device)

## 4. Compare the results and pick the best model

In [ ]:
import pandas as pd
from IPython.display import Image as Show, display
from report import plot_comparison
from run_experiments import copy_examples, select_best

results = pd.read_csv(PROJECT / "results" / "results.csv")
plot_comparison(results, PROJECT / "results" / "comparison.png")
best = select_best(PROJECT)
copy_examples(df, root, PROJECT / "examples")
print("Best experiment:", best)
display(results)
display(Show(str(PROJECT / "results" / "comparison.png")))
display(Show(str(PROJECT / "results" / f"confusion_{best}.png")))